In [ ]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import StateSpace, lsim

# Agregar directorio raíz para importar los módulos de src/
sys.path.append(os.path.abspath('..'))

from src.rom import load_lti_matrices
from src.enkf import EnsembleKalmanFilter
from src.fatigue import FatigueEstimator

In [ ]:
np.random.seed(37)
# 1. Cargar Matrices del ROM
csv_path = '../AGARD_445_6_files/dp0/SYS/MECH/modal_data_with_stress.csv'
A, B, C, D = load_lti_matrices(csv_path)

# 2. Configurar Simulación y Turbulencia de Viento (Ornstein-Uhlenbeck)
dt = 0.001  # Paso de 1 ms
t_span = np.arange(0, 2.0, dt)
n_steps = len(t_span)

u_wind = np.zeros(n_steps)
tau_viento = 0.08
sigma_viento = 12.0

for k in range(1, n_steps):
    u_wind[k] = u_wind[k-1] * np.exp(-dt/tau_viento) + \
                sigma_viento * np.sqrt(1 - np.exp(-2*dt/tau_viento)) * np.random.randn()

# 3. Estado Físico Real (Ground Truth)
sys_lti = StateSpace(A, B, C, D)
_, y_true, _ = lsim(sys_lti, U=u_wind, T=t_span)
y_disp_true = y_true[:, :3]
sigma_root_true = y_true[:, 3]

# 4. Inyección de Ruido Gaussiano a los Sensores
R_cov = np.diag([(1.5e-6)**2] * 3)
y_noisy = y_disp_true + np.random.multivariate_normal(np.zeros(3), R_cov, size=n_steps)

# 5. Inicializar Filtro EnKF y Estimador de Fatiga
Q_cov = np.eye(40) * 1e-7
enkf = EnsembleKalmanFilter(A, B, C[:3, :], Q_cov, R_cov, num_ensembles=50, dt=dt)
fatigue = FatigueEstimator(C_sn=1e12, m_sn=3.0)

C_stress = C[3, :].reshape(1, -1)
sigma_est_mean = np.zeros(n_steps)
sigma_est_std = np.zeros(n_steps)
latencies = []

# 6. Bucle de Estimación
for k in range(n_steps):
    t_start = time.perf_counter()
    
    enkf.predict(u_wind[k])
    enkf.update(y_noisy[k])
    
    latencies.append((time.perf_counter() - t_start) * 1000)
    
    sigma_ens = (C_stress @ enkf.X).ravel()
    sigma_est_mean[k] = np.mean(sigma_ens)
    sigma_est_std[k] = np.std(sigma_ens)

# 7. Evaluación de Daño Acumulado
damage = fatigue.compute_cumulative_damage(sigma_est_mean)
rul = fatigue.remaining_useful_life_percent(damage)

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)

# Panel 1: Sensores Ruidosos vs Físicos
axs[0].plot(t_span, y_noisy[:, 0] * 1e3, 'r', alpha=0.4, label='Sensor 1 (Ruidoso)')
axs[0].plot(t_span, y_disp_true[:, 0] * 1e3, 'k', label='Sensor 1 (Real)', linewidth=1.2)
axs[0].set_ylabel('Desplazamiento [mm]')
axs[0].set_title('Gemelo Digital AGARD 445.6 - Monitoreo en Tiempo Real', fontsize=12)
axs[0].grid(True, linestyle=':', alpha=0.6)
axs[0].legend(loc='upper right')

# Panel 2: Reconstrucción de Esfuerzos en Raíz e Incertidumbre
axs[1].plot(t_span, sigma_est_mean / 1e6, 'b', label='Estimado (EnKF)', linewidth=1.2)
#axs[1].fill_between(
#    t_span, 
#    (sigma_est_mean - 2 * sigma_est_std) / 1e6, 
#    (sigma_est_mean + 2 * sigma_est_std) / 1e6, 
#    color='blue', alpha=0.25, label='Incertidumbre $\pm 2\sigma$'
#)
axs[1].plot(t_span, sigma_root_true / 1e6, 'k--', label='Real (Ansys FEA)', linewidth=1.5)
axs[1].set_ylabel('Esfuerzo Raíz [MPa]')
axs[1].grid(True, linestyle=':', alpha=0.6)
axs[1].legend(loc='upper right')

# Panel 3: Benchmark de Latencia (< 15 ms)
axs[2].plot(t_span, latencies, 'g', alpha=0.7, label='Latencia por iteración')
axs[2].axhline(15.0, color='r', linestyle='--', label='Límite de Tiempo Real (15 ms)')
axs[2].set_ylabel('Latencia [ms]')
axs[2].set_xlabel('Tiempo [s]')
axs[2].grid(True, linestyle=':', alpha=0.6)
axs[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"=== METRICAS DEL GEMELO DIGITAL ===")
print(f"Latencia Media: {np.mean(latencies):.3f} ms")
print(f"Daño Acumulado (Palmgren-Miner): {damage:.6e}")
print(f"Vida Útil Restante (RUL): {rul:.2f}%")

In [ ]:
plt.plot(t_span, u_wind)

In [ ]:
import pandas as pd

# Creamos una tabla con dos columnas: Tiempo y Fuerza
df_ansys = pd.DataFrame({
    'Time': t_span,
    'Force': u_wind
})

# Copia los datos directamente al portapapeles de Windows (listos para pegar)
df_ansys.to_clipboard(index=False, header=False, decimal=',')
print("¡Datos listos en el portapapeles! Ya puedes ir a Ansys.")


In [ ]:
n_steps

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import StateSpace, lsim
from sklearn.metrics import mean_squared_error

np.random.seed(37)

# 1. Cargar Matrices del ROM de Ansys
csv_path = '../AGARD_445_6_files/dp0/SYS/MECH/modal_data_with_stress.csv'
A, B, C, D = load_lti_matrices(csv_path)

# 2. Cargar los datos de la Verdad Terrena exportados de Ansys
ansys_file = 'ansys_displacement_sensor.txt' 
ansys_data = pd.read_csv(ansys_file, sep='\t') 
ansys_data['Time [s]'] = pd.to_numeric(
    ansys_data['Time [s]'].str.replace(',', '.'), 
    errors='coerce'
)
ansys_data['Deformation Probe (Z) [m]'] = pd.to_numeric(
    ansys_data['Deformation Probe (Z) [m]'].str.replace(',', '.'), 
    errors='coerce'
)
t_ansys = ansys_data['Time [s]'].values
disp_ansys_true = ansys_data['Deformation Probe (Z) [m]'].values

# 3. Configurar la misma entrada de viento (Ornstein-Uhlenbeck)
dt = 0.001  # Paso de 1 ms
t_span = np.arange(0, 2.0, dt)
n_steps = len(t_span)

u_wind = np.zeros(n_steps)
tau_viento = 0.08
sigma_viento = 12.0

for k in range(1, n_steps):
    u_wind[k] = u_wind[k-1] * np.exp(-dt/tau_viento) + \
                sigma_viento * np.sqrt(1 - np.exp(-2*dt/tau_viento)) * np.random.randn()

# 4. Simulación en Bucle Abierto del ROM (State Space puro)
sys_lti = StateSpace(A, B, C, D)
_, y_rom, _ = lsim(sys_lti, U=u_wind, T=t_span)

# 5. Extraer la salida correspondiente al nodo de la esquina
# (Ajusta este 'idx_esquina' según la posición del nodo en las salidas de tu matriz C)
idx_esquina = 2 
disp_rom_esquina = y_rom[:, idx_esquina]

# Sincronizar longitudes para la evaluación
min_len = min(len(t_ansys), len(disp_rom_esquina))
rmse = np.sqrt(mean_squared_error(disp_ansys_true[:min_len], disp_rom_esquina[:min_len]))

# 6. Gráfica de Validación Directa (ROM vs Ansys Transient)
plt.figure(figsize=(10, 5))
plt.plot(t_span[:min_len], disp_rom_esquina[:min_len] * 1e3, 'b-', label='ROM Bucle Abierto (Python)', linewidth=1.2)
plt.plot(t_ansys[:min_len], disp_ansys_true[:min_len] * 1e3, 'k--', label='Verdad Terrena Ansys (Transient)', linewidth=1.5)
plt.title('Validación de Dinámica Estructural: ROM vs Ansys FEA (Esquina del Ala)', fontsize=12)
plt.ylabel('Desplazamiento Z [mm]')
plt.xlabel('Tiempo [s]')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f"=== MÉTRICA DE VALIDACIÓN EN BUCLE ABIERTO ===")
print(f"RMSE (ROM vs Ansys): {rmse*1e3:.4f} mm")

In [ ]:
# 1. Simulación en Bucle Abierto: lsim devuelve (tiempo, salidas, estados)
sys_lti = StateSpace(A, B, C, D)
t_out, y_rom, x_states = lsim(sys_lti, U=u_wind, T=t_span)

# 2. Extraer el nodo de la esquina
# Si tu matriz C exportada desde Ansys incluye las salidas de toda la malla,
# solo debes identificar qué fila de C corresponde a la esquina y extraerla:
# (Ejemplo: supongamos que el índice de la esquina en C es la fila 'idx_esquina')
idx_esquina = 3  # Cambia esto por el índice real de la fila de ese nodo en C
C_esquina = C[idx_esquina, :].reshape(1, -1)
D_esquina = D[idx_esquina] if np.isscalar(D) else D[idx_esquina, :]

# Reconstruimos la señal física de ese nodo usando los estados x_states
disp_rom_esquina = (C_esquina @ x_states.T).ravel() + (D_esquina * u_wind if np.isscalar(D_esquina) else 0)

# Sincronizar longitudes para la evaluación del error
min_len = min(len(t_ansys), len(disp_rom_esquina))
rmse = np.sqrt(mean_squared_error(disp_ansys_true[:min_len], disp_rom_esquina[:min_len]))

# 3. Gráfica de Validación Directa (ROM vs Ansys Transient)
plt.figure(figsize=(10, 5))
plt.plot(t_span[:min_len], disp_rom_esquina[:min_len] * 1e3, 'b-', label='ROM Reconstruido (Python)', linewidth=1.2)
plt.plot(t_ansys[:min_len], disp_ansys_true[:min_len] * 1e3, 'k--', label='Verdad Terrena Ansys (Transient)', linewidth=1.5)
plt.title('Validación de Dinámica Estructural: ROM vs Ansys FEA (Esquina del Ala)', fontsize=12)
plt.ylabel('Desplazamiento Z [mm]')
plt.xlabel('Tiempo [s]')
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"=== MÉTRICA DE VALIDACIÓN EN BUCLE ABIERTO ===")
print(f"RMSE (ROM vs Ansys): {rmse*1e3:.4f} mm")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import StateSpace, lsim
from sklearn.metrics import mean_squared_error

np.random.seed(37)

# 1. Cargar Matrices del ROM de Ansys
csv_path = '../AGARD_445_6_files/dp0/SYS/MECH/modal_data_with_stress.csv'
A, B, C, D = load_lti_matrices(csv_path)

# 2. Cargar la malla completa y las formas modales de Ansys
csv_mesh_path = '../AGARD_445_6_files/dp0/SYS/MECH/wing_full_mesh.csv'
df_mesh = pd.read_csv(csv_mesh_path, skip_blank_lines=True, skipinitialspace=True)
df_mesh.columns = df_mesh.columns.str.strip()

X_nodes = df_mesh['X'].values
Y_nodes = df_mesh['Y'].values

# Extraer todas las columnas de formas modales en Z (ej. Phi1_Z hasta Phi20_Z)
phi_cols = [col for col in df_mesh.columns if 'Phi' in col and '_Z' in col]
Phi_real = df_mesh[phi_cols].values  # Matriz de forma modal de toda la malla

# 3. Localizar el nodo de la esquina que mediste en Ansys dentro de la malla
# (Ingresa aquí las coordenadas aproximadas X e Y de la esquina donde pusiste la sonda)
x_target, y_target = 1.13, 0.762  # <--- Reemplaza con las coordenadas reales de tu esquina en Ansys
distancias = np.sqrt((X_nodes - x_target)**2 + (Y_nodes - y_target)**2)
idx_nodo_esquina = np.argmin(distancias)

# Extraer el vector de deformación modal solo para ese nodo específico de la esquina
phi_esquina_nodo = Phi_real[idx_nodo_esquina, :]

# 4. Cargar los datos de la Verdad Terrena de la esquina exportados de Ansys
ansys_file = 'ansys_displacement_sensor.txt' 
ansys_data = pd.read_csv(ansys_file, sep='\t') 
ansys_data['Time [s]'] = pd.to_numeric(ansys_data['Time [s]'].str.replace(',', '.'), errors='coerce')
ansys_data['Deformation Probe (Z) [m]'] = pd.to_numeric(ansys_data['Deformation Probe (Z) [m]'].str.replace(',', '.'), errors='coerce')

t_ansys = ansys_data['Time [s]'].values
disp_ansys_true = ansys_data['Deformation Probe (Z) [m]'].values

# 5. Configurar la entrada de viento (Ornstein-Uhlenbeck)
dt = 0.001  
t_span = np.arange(0, 2.0, dt)
n_steps = len(t_span)

u_wind = np.zeros(n_steps)
tau_viento = 0.08
sigma_viento = 12.0

for k in range(1, n_steps):
    u_wind[k] = u_wind[k-1] * np.exp(-dt/tau_viento) + \
                sigma_viento * np.sqrt(1 - np.exp(-2*dt/tau_viento)) * np.random.randn()

# 6. Simulación en Bucle Abierto del ROM (Obteniendo estados x)
sys_lti = StateSpace(A, B, C, D)
t_out, y_rom, x_states = lsim(sys_lti, U=u_wind, T=t_span)

# Extraer las coordenadas modales q(t) de los estados del sistema 
# (Asumiendo que las primeras N_modos columnas de x corresponden a q)
num_modos = len(phi_cols)
q_t = x_states[:, :num_modos]  # Forma: (n_steps, num_modos)

# 7. Reconstruir el desplazamiento de la esquina usando superposición modal: Z = Phi * q(t)
disp_rom_esquina = q_t @ phi_esquina_nodo

# Sincronizar longitudes para la evaluación
min_len = min(len(t_ansys), len(disp_rom_esquina))
rmse = np.sqrt(mean_squared_error(disp_ansys_true[:min_len], disp_rom_esquina[:min_len]))

# 8. Gráfica de Validación Directa (ROM Reconstruido vs Ansys Transient)
plt.figure(figsize=(10, 5))
plt.plot(t_span[:min_len], disp_rom_esquina[:min_len] * 1e3, 'b-', label='ROM Reconstruido (Esquina)', linewidth=1.2)
plt.plot(t_ansys[:min_len], disp_ansys_true[:min_len] * 1e3, 'k--', label='Verdad Terrena Ansys (Transient)', linewidth=1.5)
plt.title('Validación Cruzada: Reconstrucción Modal en Esquina vs Ansys FEA', fontsize=12)
plt.ylabel('Desplazamiento Z [mm]')
plt.xlabel('Tiempo [s]')
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"=== MÉTRICA DE VALIDACIÓN EN BUCLE ABIERTO (ESQUINA) ===")
print(f"RMSE (ROM vs Ansys): {rmse*1e3:.4f} mm")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.signal import StateSpace, lsim

np.random.seed(37)

# 1. Cargar Matrices del ROM de Ansys
csv_path = '../AGARD_445_6_files/dp0/SYS/MECH/modal_data_with_stress.csv'
A, B, C, D = load_lti_matrices(csv_path)

# 2. Cargar Malla y Modos Reales de Ansys
csv_mesh_path = '../AGARD_445_6_files/dp0/SYS/MECH/wing_full_mesh.csv'
df_mesh = pd.read_csv(csv_mesh_path, skip_blank_lines=True, skipinitialspace=True)
df_mesh.columns = df_mesh.columns.str.strip()

# Submuestreo de la malla (puedes ajustar el paso si deseas más velocidad)
df_sub = df_mesh.iloc[::1, :]

X_nodes = df_sub['X'].values
Y_nodes = df_sub['Y'].values
Z_nodes_base = df_sub['Z'].values if 'Z' in df_sub.columns else np.zeros_like(X_nodes)

# Extraer automáticamente todas las columnas modales en Z de la malla
phi_cols = [col for col in df_sub.columns if 'Phi' in col and '_Z' in col]
Phi_real = df_sub[phi_cols].values  # Matriz de formas modales de toda la malla

AMP_SCALE = 5.0  # Escala visual para apreciar claramente la flexión

# 3. Configurar simulación temporal y turbulencia de viento (Ornstein-Uhlenbeck)
dt = 0.001  
t_span = np.arange(0, 2.0, dt)
n_steps = len(t_span)

u_wind = np.zeros(n_steps)
tau_viento = 0.08
sigma_viento = 12.0

for k in range(1, n_steps):
    u_wind[k] = u_wind[k-1] * np.exp(-dt/tau_viento) + \
                sigma_viento * np.sqrt(1 - np.exp(-2*dt/tau_viento)) * np.random.randn()

# 4. Simulación en Bucle Abierto del ROM (Extrayendo estados x)
sys_lti = StateSpace(A, B, C, D)
t_out, y_rom, x_states = lsim(sys_lti, U=u_wind, T=t_span)

# Extraer las coordenadas modales reales q(t) desde los estados internos
num_modos = len(phi_cols)
q_t = x_states[:, :num_modos]  # Coordenadas modales correspondientes a la dinámica del viento

# Submuestrear frames para que la animación sea fluida y rápida al renderizar
frame_step = 10  # Toma 1 de cada 10 pasos de tiempo
t_vec = t_span[::frame_step]
q_animation = q_t[::frame_step, :]

# 5. Configuración de la figura 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

norm = plt.Normalize(vmin=-0.015, vmax=0.015)
mappable = plt.cm.ScalarMappable(norm=norm, cmap='jet')
cbar = fig.colorbar(mappable, ax=ax, shrink=0.7, pad=0.1)
cbar.set_label('Deformación Z Real [m]', fontsize=10)

def update_3d_vibration(frame):
    ax.clear()
    
    q_k = q_animation[frame, :]
    Z_physical = Phi_real @ q_k  # Superposición modal usando los estados del ROM
    Z_render = Z_nodes_base + Z_physical * AMP_SCALE
    
    surf = ax.plot_trisurf(
        X_nodes, Y_nodes, Z_render,
        cmap='jet', norm=norm,
        edgecolor='none', alpha=0.95
    )
    
    ax.set_zlim(-0.1, 0.1)
    ax.set_title(
        f'Gemelo Digital AGARD 445.6 - Reconstrucción 3D (ROM)\n'
        f'Tiempo: {t_vec[frame]:.2f} s | Escala Visual: {AMP_SCALE:.1f}x',
        fontsize=11
    )
    ax.set_xlabel('Cuerda X [m]')
    ax.set_ylabel('Envergadura Y [m]')
    ax.set_zlabel('Desplazamiento Z [m]')
    ax.view_init(elev=20, azim=-125)

print('Generando animación 3D con la respuesta dinámica real del ROM...')
ani = animation.FuncAnimation(
    fig, update_3d_vibration, frames=len(t_vec), interval=30, blit=False
)

# Guardar como GIF animado
ani.save('wing_3d_vibration_rom.gif', writer='pillow', fps=30)
print('¡GIF guardado con éxito como wing_3d_vibration_rom.gif!')

plt.show()